# Import Library

In [2]:
import mlflow
import numpy as np
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Dataset

In [3]:
df=pd.read_csv("https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/refs/heads/main/tweet_emotions.csv")
df.head()

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


- Drop 'tweet_id'

In [4]:
df.drop(columns="tweet_id",inplace=True)

- Data Preprocessing

In [5]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def lemmatization(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

def remove_stop_words(text):
    return " ".join([word for word in str(text).split() if word not in stop_words])

def removing_numbers(text):
    return "".join([char for char in text if not char.isdigit()])

def lower_case(text):
    return " ".join([word.lower() for word in text.split()])

def remove_punctuation(text):
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    return text.strip()

def remove_urls(text):
    return re.sub(r"http\S+", "", text).strip()


def normalize_text(df: pd.DataFrame) -> pd.DataFrame:
    try:
        print("Starting text normalization")

        df['content'] = df['content'].astype(str)

        df['content'] = df['content'].apply(remove_urls)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_punctuation)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(lemmatization)

        print("Text normalization completed")
        return df

    except Exception as e:
        print(f"Error in text normalization: {e}")
        raise

In [6]:
df=normalize_text(df)

Starting text normalization
Text normalization completed


In [7]:
df.head()

,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin n bed headache ughhhhwaitin call
2,sadness,funeral ceremonygloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [8]:
df['sentiment'].value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

- we just focus on two sentiment 'happiness' and 'sadness'

In [9]:
df=df[df['sentiment'].isin(['happiness','sadness'])]

In [10]:
df.head()


,sentiment,content
1,sadness,layin n bed headache ughhhhwaitin call
2,sadness,funeral ceremonygloomy friday
6,sadness,sleep im thinking old friend want he married d...
8,sadness,charviray charlene love miss
9,sadness,kelcouch im sorry least friday


- Replace 'sadness' with 0 and 'happiness' with 1

In [11]:
df['sentiment'].replace({'sadness':0,'happiness':1},inplace=True)

C:\Users\hp\AppData\Local\Temp\ipykernel_15764\2140401980.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'].replace({'sadness':0,'happiness':1},inplace=True)


In [12]:
df.sample(5)

,sentiment,content
25540,1,ingridholliday good morning
35164,1,feelin bit dangerous watch tha vids
18245,0,kdpartak ordered new blackberry arrive tues ma...
38875,0,today im lonley girl guitar
18716,0,fizzythoughts boo work beatwittyparty least yo...


- Extract X and y

In [13]:
X=df['content'] #input
y=df['sentiment'] #target

- Apply CountVectorizer

In [14]:
countvector=CountVectorizer(max_features=1000)
X=countvector.fit_transform(X)

In [15]:
X

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 52959 stored elements and shape (10374, 1000)>

- Split the data

In [16]:
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

- connect with dagshub

In [17]:
import dagshub
import mlflow
mlflow.set_tracking_uri("https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/")
dagshub.init(repo_owner='rahulpatel16092005', repo_name='mlops-mini-project', mlflow=True)

Accessing as rahulpatel16092005

Initialized MLflow to track repo "rahulpatel16092005/mlops-mini-project"

Repository rahulpatel16092005/mlops-mini-project initialized!

In [18]:
mlflow.set_experiment("Logistic Regression Baseline model")

<Experiment: artifact_location='mlflow-artifacts:/ddba2e912cdf43c19c282cc8d33078d2', creation_time=1774767750213, experiment_id='0', last_update_time=1774767750213, lifecycle_stage='active', name='Logistic Regression Baseline model', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [20]:
with mlflow.start_run():

    mlflow.log_param("Vectorizer","Bag of words")
    mlflow.log_param("max_features",1000)
    mlflow.log_param("test_size",0.2)

    model = LogisticRegression()
    model.fit(x_train, y_train)

    mlflow.log_param("model", "Logistic Regression")

    y_pred = model.predict(x_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(model, "Logistic Regression baseline model")

    notebook_path="exp1_baseline_model.ipynb"
    import os
    os.system(f"jupyter nbconvert --to script {notebook_path}")
    mlflow.log_artifact(notebook_path)

2026/03/29 12:45:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 12:45:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run masked-mink-934 at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/0/runs/e87585a2d1d644968237ed4b93974a05
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/0
